# The book as a fold

The answers to Section 4 of the sample questions, `documentation/tex/questions`, run.
The questions concern Section 1.2 of the lecture notes: the fold of equation (1.2.1), its
implementation in Section 1.2.3, and the LOBSTER message file of Section 1.2.5.

Each question's code is copied here verbatim, wrapped in a function `snippet` where it prints:
the name `run` is the driver of `unito26.lob.session`, which several snippets call.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
Prices are integer counts of ticks.

In [ ]:
import contextlib
import io
import timeit

import numpy as np
import pandas as pd
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import ExponentialHawkes, HawkesParams
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")

In [ ]:
import sys
from functools import reduce

from unito26.lob.delta_log import DeltaLog
from unito26.lob.session import run

## `question.foldByReduceOverApply`: three folds, two of them by `reduce`

In [ ]:
def fold_a(book, messages):
    for message in messages:
        book.apply(message, record=False)
    return book

def transition(book, message):
    after = book.copy()
    after.apply(message, record=False)
    return after

def fold_b(book, messages):
    return reduce(transition, messages, book)

def fold_c(book, messages):
    return reduce(lambda b, m: b.apply(m, record=False), messages, book)

In [ ]:
def snippet(fold):
    stream = [limit_order(1.0, 50, 1001, BUY), market_order(2.0, 80, SELL)]
    book = AggregateBook.from_levels({1000: 100}, {1002: 120})
    final = fold(book, stream)
    print(final.best_bid_size, book.best_bid_size)


a, b, c = outcome(snippet, fold_a), outcome(snippet, fold_b), outcome(snippet, fold_c)
print("A:", a, "   B:", b, "   C:", c)
verify("question.foldByReduceOverApply", "C", [
    a == b == "70 70" and c == "AttributeError",
    a == "70 70" and b == "70 100" and c == "70 70",
    a == "70 70" and b == "70 100" and c == "AttributeError",
    a == b == "70 100" and c == "AttributeError",
])

## `question.copyingFoldAgainstInPlaceFold`: a copy of the book per message

Folds A and B of the previous question are timed on one stream of 1000 messages: limit orders
and withdrawals of 10 shares at the five prices nearest each touch, none crossing the spread.
Each fold starts from a book with $\ell$ occupied prices on each side, 100 shares at each.
The choices are read off the ratio of the two times at each $\ell$, and off the growth of
each time from $\ell = 10$ to $\ell = 1000$.

In [ ]:
def near_the_touch(n, rng):
    messages = []
    for t in range(n):
        direction = BUY if rng.random() < 0.5 else SELL
        offset = int(rng.integers(0, 5))
        price = 1000 - offset if direction == BUY else 1002 + offset
        if rng.random() < 0.5:
            messages.append(limit_order(float(t), 10, price, direction))
        else:
            messages.append(withdrawal(float(t), 10, price, direction))
    return messages


def deep_book(levels):
    return AggregateBook.from_levels(
        {1000 - k: 100 for k in range(levels)}, {1002 + k: 100 for k in range(levels)}
    )


def seconds_per_message(fold, levels, messages):
    total = min(timeit.repeat(lambda: fold(deep_book(levels), messages), number=1, repeat=3))
    return total / len(messages)


messages = near_the_touch(1_000, np.random.default_rng(4))
timings = {}
for levels in (10, 100, 1_000):
    timings[levels] = (
        seconds_per_message(fold_a, levels, messages),
        seconds_per_message(fold_b, levels, messages),
    )
    print(f"l = {levels:>5}:  in place (A) {timings[levels][0] * 1e6:8.2f} us   "
          f"copying (B) {timings[levels][1] * 1e6:8.2f} us per message   "
          f"B over A {timings[levels][1] / timings[levels][0]:5.1f}x")

ratio = {levels: timings[levels][1] / timings[levels][0] for levels in timings}
growth_in_place = timings[1_000][0] / timings[10][0]
growth_copying = timings[1_000][1] / timings[10][1]
print(f"from l = 10 to l = 1000: A grows {growth_in_place:.1f}x, B grows {growth_copying:.1f}x")
verify("question.copyingFoldAgainstInPlaceFold", "B", [
    all(r < 2 for r in ratio.values()),
    ratio[1_000] > 10 and ratio[1_000] > 2 * ratio[10],
    all(1.5 < r < 3 for r in ratio.values()),
    0.5 < growth_copying / growth_in_place < 2,
])

## `question.recursiveFoldOverAStream`: a recursive fold and the limit on recursion

The stem states the default limit on the depth of recursion; it is checked first.

In [ ]:
assert sys.getrecursionlimit() == 1000

In [ ]:
def fold_a(book, messages):
    if not messages:
        return book
    book.apply(messages[0], record=False)
    return fold_a(book, messages[1:])

def fold_b(book, messages):
    for message in messages:
        book.apply(message, record=False)
    return book

In [ ]:
def snippet(fold, n):
    stream = [limit_order(float(t), 10, 1000, BUY) for t in range(n)]
    print(fold(AggregateBook(), stream).best_bid_size)


a = [outcome(snippet, fold_a, n) for n in (500, 5000)]
b = [outcome(snippet, fold_b, n) for n in (500, 5000)]
print("A:", a, "   B:", b)
verify("question.recursiveFoldOverAStream", "D", [
    a == b == ["5000", "50000"],
    a == ["RecursionError", "RecursionError"] and b == ["5000", "50000"],
    a == b == ["5000", "RecursionError"],
    a == ["5000", "RecursionError"] and b == ["5000", "50000"],
])

## `question.messagesThatCommuteInTheFold`: does the order of two messages matter

In [ ]:
def snippet():
    def final(pair):
        book = AggregateBook.from_levels(
            {1000: 100, 999: 200}, {1002: 120, 1003: 80}
        )
        for message in pair:
            book.apply(message, record=False)
        return book.bids, book.asks

    bid_1001 = limit_order(1.0, 50, 1001, BUY)
    ask_1001 = limit_order(1.0, 40, 1001, SELL)
    cancel = withdrawal(1.0, 30, 1000, BUY)
    sell_80 = market_order(1.0, 80, SELL)
    sell_30 = market_order(1.0, 30, SELL)
    pairs = {"i": (bid_1001, ask_1001), "ii": (cancel, sell_80),
             "iii": (bid_1001, sell_30)}
    for name, (m, n) in pairs.items():
        print(name, final((m, n)) == final((n, m)))


lines = outcome(snippet).split("\n")
print(lines)
differ = {line.split()[0] for line in lines if line.split()[1] == "False"}
verify("question.messagesThatCommuteInTheFold", "A", [
    differ == {"ii", "iii"},
    differ == {"i", "ii", "iii"},
    differ == {"iii"},
    differ == {"i", "iii"},
])

## `question.recorderAlignedWithTheOrderbookFile`: the book, a copy before, a copy after

In [ ]:
def snippet():
    book = AggregateBook.from_levels({1000: 100}, {1002: 120})
    stream = [
        limit_order(1.0, 50, 1001, BUY),
        market_order(2.0, 80, SELL),
        withdrawal(3.0, 70, 1000, BUY),
    ]
    a, b, c = [], [], []
    for message in stream:
        b.append(book.copy())
        book.apply(message, record=False)
        a.append(book)
        c.append(book.copy())
    for states in (a, b, c):
        print([state.best_bid_price for state in states])


lines = outcome(snippet).split("\n")
print(lines)
verify("question.recorderAlignedWithTheOrderbookFile", "B", [
    lines == ["[1001, 1000, None]", "[1000, 1001, 1000]", "[1001, 1000, None]"],
    lines == ["[None, None, None]", "[1000, 1001, 1000]", "[1001, 1000, None]"],
    lines == ["[None, None, None]", "[1001, 1000, None]", "[1001, 1000, None]"],
    lines == ["[1000, 1000, 1000]", "[1000, 1001, 1000]", "[1001, 1000, 1000]"],
])

## `question.runTwiceOverOneGenerator`: one generator, two books

In [ ]:
def snippet():
    bids, asks = {1000: 100}, {1002: 120}
    book_a = AggregateBook.from_levels(bids, asks)
    book_b = AggregateBook.from_levels(bids, asks)
    messages = [limit_order(1.0, 50, 1001, BUY), market_order(2.0, 80, SELL)]
    stream = (message for message in messages)
    first, second = run(book_a, stream), run(book_b, stream)
    print(first["traded"], second["traded"], book_b.best_bid_size)


printed = outcome(snippet)
print(printed)
verify("question.runTwiceOverOneGenerator", "C", [
    printed == "80 80 70",
    printed == "80 0 70",
    printed == "80 0 100",
    printed == "KeyError",
])

## `question.spreadsRefoldedFromEmpty`: a fold per state against one fold

Both are timed on the same stream at each length $n$; the book never holds more than ten
prices.  The choices are read off the equality of the two lists and off the growth of the
ratio of the two times with $n$.

In [ ]:
def spreads_a(messages):
    spreads = []
    for k in range(1, len(messages) + 1):
        book = AggregateBook()
        run(book, messages[:k])
        spreads.append(book.spread)
    return spreads

def spreads_b(messages):
    book, spreads = AggregateBook(), []
    for message in messages:
        book.apply(message, record=False)
        spreads.append(book.spread)
    return spreads

In [ ]:
def seconds(function, messages):
    return min(timeit.repeat(lambda: function(messages), number=1, repeat=3))


ratios, equal = {}, True
for n in (250, 500, 1_000):
    messages = near_the_touch(n, np.random.default_rng(n))
    equal = equal and spreads_a(messages) == spreads_b(messages)
    a, b = seconds(spreads_a, messages), seconds(spreads_b, messages)
    ratios[n] = a / b
    print(f"n = {n:>5}:  A {a * 1e3:8.1f} ms   B {b * 1e3:6.2f} ms   A over B {ratios[n]:6.0f}x")

growth = ratios[1_000] / ratios[250]
print(f"lists equal: {equal};  the ratio grows {growth:.1f}x from n = 250 to n = 1000")
verify("question.spreadsRefoldedFromEmpty", "D", [
    equal and 0.5 < growth < 2,
    all(r < 1 for r in ratios.values()),
    not equal,
    equal and growth > 2.5 and ratios[1_000] > 5,
])

## `question.deltaLogReplayIsIdempotent`: a repeated or lost record, as a message and as a delta

In [ ]:
def snippet():
    opening = ({1000: 100, 999: 200}, {1002: 120})
    stream = [
        limit_order(1.0, 50, 1000, BUY),
        market_order(2.0, 120, SELL),
        withdrawal(3.0, 40, 999, BUY),
    ]
    log = DeltaLog.record(AggregateBook.from_levels(*opening), stream)
    deltas = [delta for _, _, delta in log.entries]

    def replay_messages(messages):
        book = AggregateBook.from_levels(*opening)
        for message in messages:
            book.apply(message, record=False)
        return book.bids

    def replay_deltas(deltas):
        book = AggregateBook.from_levels(*opening)
        for delta in deltas:
            book.set_size(delta.side, delta.price, delta.resting)
        return book.bids

    clean = replay_messages(stream)
    print(replay_messages(stream[:1] + stream) == clean,
          replay_deltas(deltas[:1] + deltas) == clean,
          replay_deltas(deltas[1:]) == clean,
          replay_deltas(deltas[:-1]) == clean)


printed = outcome(snippet)
print(printed)
verify("question.deltaLogReplayIsIdempotent", "A", [
    printed == "False True True False",
    printed == "False False False False",
    printed == "False True False False",
    printed == "False True True True",
])

## `question.recordedStreamFoldedFromEmpty`: an opening book, an empty one, a strict one

The fold raises part-way, so what it printed before is kept beside the exception.

In [ ]:
def printed_and_raised(function):
    """The lines a call prints, and the name of the exception it raises or None."""
    printed = io.StringIO()
    raised = None
    try:
        with contextlib.redirect_stdout(printed):
            function()
    except Exception as error:
        raised = type(error).__name__
    return printed.getvalue().strip().split("\n"), raised


def snippet():
    stream = [
        withdrawal(1.0, 40, 1000, BUY),
        market_order(2.0, 80, SELL),
        limit_order(3.0, 30, 1001, SELL),
    ]

    def fold(book):
        for message in stream:
            book.apply(message, record=False)
        print(book.best_bid_price, book.best_ask_price)

    fold(AggregateBook.from_levels({1000: 100, 999: 50}, {1002: 120}))
    fold(AggregateBook())
    fold(AggregateBook(strict=True))


lines, raised = printed_and_raised(snippet)
print(lines, raised)
verify("question.recordedStreamFoldedFromEmpty", "B", [
    lines == ["999 1001", "None 1001", "None 1001"] and raised is None,
    lines == ["999 1001", "None 1001"] and raised == "ValueError",
    lines == ["999 1001", "None 0"] and raised == "ValueError",
    lines == ["999 1001"] and raised == "ValueError",
])

## `question.lobsterExecutionsReplayed`: an execution as a withdrawal or as a market order

In [ ]:
rows = [(1.0, 60, 1000, BUY), (1.0, 40, 1000, BUY), (1.0, 50, 999, BUY)]

def replay_a(t, size, price, d):
    return withdrawal(t, size, price, d)

def replay_b(t, size, price, d):
    return market_order(t, size, -d)

def replay_c(t, size, price, d):
    return market_order(t, size, d)

In [ ]:
def snippet(replay):
    book = AggregateBook.from_levels({1000: 100, 999: 200}, {1002: 120})
    tally = run(book, [replay(*row) for row in rows])
    print(book.best_bid_price, book.best_bid_size, tally["traded"])


a, b, c = outcome(snippet, replay_a), outcome(snippet, replay_b), outcome(snippet, replay_c)
print("A:", a, "   B:", b, "   C:", c)
verify("question.lobsterExecutionsReplayed", "C", [
    a == b == "999 150 150" and c == "1002 30 120",
    a == "999 150 0" and b == "1002 30 120" and c == "999 150 150",
    a == "999 150 0" and b == "999 150 150" and c == "1002 30 120",
    a == "999 150 0" and b == "999 150 150" and c == "1000 100 120",
])

## `question.lobsterHaltRowsReplayed`: a hidden execution and a halt, converted to messages

In [ ]:
def to_message(t, kind, size, price, d):
    if kind in (2, 3):
        return withdrawal(t, size, price, d)
    if kind in (4, 5):
        return market_order(t, size, -d)
    return limit_order(t, size, price, d)

In [ ]:
def snippet():
    rows = [
        (34200.0, 5, 30, 1001, SELL),
        (36023.0, 7, 0, -1, SELL),
        (36323.0, 7, 0, 0, SELL),
        (36723.0, 7, 0, 1, SELL),
    ]
    book = AggregateBook.from_levels({1000: 100}, {1002: 120})
    for row in rows:
        try:
            book.apply(to_message(*row), record=False)
            print(row[3], book.best_bid_size, book.best_ask_size)
        except ValueError:
            print(row[3], "ValueError")


lines = outcome(snippet).split("\n")
print(lines)
verify("question.lobsterHaltRowsReplayed", "D", [
    lines == ["1001 70 120", "-1 ValueError", "0 70 120", "1 70 120"],
    lines == ["1001 100 90", "-1 100 90", "0 100 90", "1 100 90"],
    lines == ["1001 100 90", "-1 ValueError", "0 ValueError", "1 100 90"],
    lines == ["1001 100 90", "-1 ValueError", "0 100 90", "1 100 90"],
])